# CineBot — Domain-Oriented Chatbot for Movie Recommendations
### Capstone Project · AI/ML/DL Course (Batch 7)

**Author:** Abhishek Kumar  ·  **Repository:** [github.com/kumarabhishek7242/cin-mood-bot](https://github.com/kumarabhishek7242/cin-mood-bot)

This notebook contains the core deep-learning experiment behind CineBot: **fine-tuning DistilBERT for intent
classification** — the NLP component that lets the chatbot understand what a user wants
(`recommend`, `refine`, `more_info`, `feedback`, `greet`, `goodbye`, `oos`) before the recommender
engine selects movies.

**Notebook structure**
1. Setup & dependencies
2. Dataset loading & exploratory analysis
3. Train / Validation / Test split (70 / 15 / 15, stratified)
4. Tokenization & PyTorch datasets
5. Model architecture (DistilBERT + classification head)
6. Training (with per-epoch validation)
7. Training curves — loss & macro-F1 over epochs
8. **Test-set evaluation (separate cell, per submission guidelines)**
9. Error analysis — confusion matrix & misclassified examples
10. End-to-end inference demo (intent + entity extraction → chatbot understanding)
11. Conclusions

## 1 · Setup & Dependencies

Installs the required libraries and fixes all random seeds so results are reproducible.

In [ ]:
# Install dependencies (quiet). On Colab, transformers/torch are mostly pre-installed.
%pip install -q transformers datasets scikit-learn torchinfo matplotlib seaborn accelerate

import json, random, warnings
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Reproducibility: fix every RNG that touches training
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

## 2 · Dataset Loading & Exploratory Analysis

The intent dataset lives in the repository at `ml/data/intents_sample.jsonl` — **216 labelled utterances
across 7 intent classes**, hand-curated for the movie-recommendation domain. Each line is a JSON object:
`{"text": "...", "label": "..."}`.

The cell below loads it from the local clone if present, otherwise pulls it straight from GitHub
(so this notebook is runnable standalone, e.g. on Colab).

In [ ]:
DATA_PATH = Path("ml/data/intents_sample.jsonl")
RAW_URL = ("https://raw.githubusercontent.com/kumarabhishek7242/"
           "cin-mood-bot/main/ml/data/intents_sample.jsonl")

# Fallback: fetch the dataset from the public repo when not running inside a clone
if not DATA_PATH.exists():
    import urllib.request
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(RAW_URL, DATA_PATH)
    print("Downloaded dataset from GitHub.")

# Parse JSONL -> parallel lists of texts and labels
texts, labels = [], []
with DATA_PATH.open() as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        texts.append(row["text"])
        labels.append(row["label"])

print(f"Loaded {len(texts)} utterances")
print("Example:", texts[0], "->", labels[0])

In [ ]:
import collections
import pandas as pd

df = pd.DataFrame({"text": texts, "label": labels})
df["n_words"] = df["text"].str.split().str.len()

# Class distribution — the dataset is imbalanced (recommend dominates),
# which motivates macro-F1 (not accuracy) as the primary metric.
counts = df["label"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
counts.plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set_title("Class distribution (n=216)")
axes[0].set_ylabel("examples")
df["n_words"].plot.hist(bins=15, ax=axes[1], color="#55A868")
axes[1].set_title("Utterance length (words)")
plt.tight_layout(); plt.show()

print(counts)
print(f"\nMean utterance length: {df.n_words.mean():.1f} words "
      f"(max {df.n_words.max()}) -> max_length=64 tokens is safe")

## 3 · Train / Validation / Test Split — 70 / 15 / 15

Three-way **stratified** split so every class keeps its proportion in each subset:
- **Train (70%)** — gradient updates
- **Validation (15%)** — per-epoch model selection (best macro-F1 checkpoint)
- **Test (15%)** — held out completely; touched **once**, in Section 8

> Stratification matters here because the smallest classes (`greet`, `goodbye`) have only ~20 examples —
> a random split could leave them absent from validation entirely.

In [ ]:
from sklearn.model_selection import train_test_split

label_list = sorted(set(labels))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
y = np.array([label2id[l] for l in labels])

# First carve out the 15% test set, then split the rest 70/15 (≈ 0.1765 of remainder)
X_tmp, X_test, y_tmp, y_test = train_test_split(
    texts, y, test_size=0.15, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.15 / 0.85, random_state=SEED, stratify=y_tmp)

print(f"train={len(X_train)}  val={len(X_val)}  test={len(X_test)}")
print("classes:", label_list)

## 4 · Tokenization & PyTorch Datasets

DistilBERT uses WordPiece tokenization. We pad/truncate to **64 tokens** — comfortably above the longest
utterance observed in the EDA — and wrap everything in a minimal `torch.utils.data.Dataset` so the
HuggingFace `Trainer` can batch it.

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "distilbert-base-uncased"
MAX_LEN = 64

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

class IntentDataset(torch.utils.data.Dataset):
    """Tokenizes utterances once up-front; returns dicts the Trainer expects."""
    def __init__(self, texts, ys):
        self.enc = tokenizer(list(texts), truncation=True,
                             padding="max_length", max_length=MAX_LEN,
                             return_tensors="pt")
        self.ys = torch.tensor(ys, dtype=torch.long)
    def __len__(self):
        return len(self.ys)
    def __getitem__(self, i):
        return {"input_ids": self.enc["input_ids"][i],
                "attention_mask": self.enc["attention_mask"][i],
                "labels": self.ys[i]}

train_ds = IntentDataset(X_train, y_train)
val_ds   = IntentDataset(X_val,   y_val)
test_ds  = IntentDataset(X_test,  y_test)

# Sanity check: round-trip one example through the tokenizer
sample = train_ds[0]
print("input_ids shape:", sample["input_ids"].shape)
print("decoded:", tokenizer.decode(sample["input_ids"], skip_special_tokens=True))

## 5 · Model Architecture

**DistilBERT** (Sanh et al., 2019) is a knowledge-distilled BERT: 6 transformer layers, 12 attention heads,
hidden size 768 — **~66M parameters**, 40% smaller and 60% faster than BERT-base while keeping ~97% of its
language-understanding performance. That trade-off suits a latency-sensitive chatbot.

`DistilBertForSequenceClassification` adds on top of the encoder:
`[CLS] pooled output → pre-classifier Linear(768→768) + ReLU → Dropout(0.2) → Linear(768→7)`

The cell below prints the full layer table (the `model.summary()`-equivalent for PyTorch, via `torchinfo`).

In [ ]:
from transformers import DistilBertForSequenceClassification
from torchinfo import summary

model = DistilBertForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(label_list),
    label2id=label2id,
    id2label=id2label,
).to(DEVICE)

# Layer-by-layer architecture table (rubric: model.summary() equivalent)
summary(model,
        input_data={"input_ids": torch.zeros(1, MAX_LEN, dtype=torch.long, device=DEVICE),
                    "attention_mask": torch.ones(1, MAX_LEN, dtype=torch.long, device=DEVICE)},
        depth=3, col_names=("output_size", "num_params"))

## 6 · Training

**Hyperparameters** (carried over from the repo's `app/scripts/train_intent.py`):

| Hyperparameter | Value | Rationale |
|---|---|---|
| Optimizer | AdamW | Trainer default; decoupled weight decay suits transformers |
| Learning rate | 2e-5 | Standard fine-tuning LR for BERT-family models |
| Batch size | 16 | Fits easily in memory at seq-len 64 |
| Epochs | 6 | Small dataset → a couple extra epochs; best checkpoint selected on val F1 |
| Warmup ratio | 0.1 | Stabilises early updates of pretrained weights |
| Weight decay | 0.01 | Light regularisation against overfitting 151 train examples |
| Loss | Cross-entropy | Multi-class single-label objective (built into the model head) |
| Callback | `load_best_model_at_end` | Restores the epoch with best **validation macro-F1** |

**Metric choice:** macro-F1 — every class counts equally despite the imbalance shown in the EDA.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from transformers import Trainer, TrainingArguments

EPOCHS, BATCH, LR = 6, 16, 2e-5

def compute_metrics(pred):
    """Macro precision/recall/F1 — robust to the class imbalance."""
    preds = pred.predictions.argmax(-1)
    p, r, f1, _ = precision_recall_fscore_support(
        pred.label_ids, preds, average="macro", zero_division=0)
    return {"precision": float(p), "recall": float(r), "f1_macro": float(f1)}

args = TrainingArguments(
    output_dir="ckpt",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",          # validate after every epoch
    save_strategy="epoch",
    load_best_model_at_end=True,    # restore best-val-F1 checkpoint
    metric_for_best_model="f1_macro",
    logging_strategy="epoch",
    seed=SEED,
    report_to=[],
)

trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=val_ds,
                  compute_metrics=compute_metrics)

train_result = trainer.train()
print(f"\nTraining done in {train_result.metrics['train_runtime']:.0f}s "
      f"| final train loss {train_result.metrics['train_loss']:.4f}")

## 7 · Training Curves — Loss & Macro-F1 over Epochs

Extracted from `trainer.state.log_history`. Watch for the gap between train and validation loss —
a widening gap signals overfitting (expected to start early on only 151 training examples,
which is exactly why we checkpoint on validation F1).

In [ ]:
hist = trainer.state.log_history

train_loss = [(h["epoch"], h["loss"])      for h in hist if "loss" in h and "eval_loss" not in h]
val_loss   = [(h["epoch"], h["eval_loss"]) for h in hist if "eval_loss" in h]
val_f1     = [(h["epoch"], h["eval_f1_macro"]) for h in hist if "eval_f1_macro" in h]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(*zip(*train_loss), "o-", label="train loss")
axes[0].plot(*zip(*val_loss),   "s-", label="validation loss")
axes[0].set(title="Loss over epochs", xlabel="epoch", ylabel="cross-entropy loss")
axes[0].legend()

axes[1].plot(*zip(*val_f1), "s-", color="#C44E52", label="validation macro-F1")
axes[1].set(title="Validation macro-F1 over epochs", xlabel="epoch", ylabel="F1", ylim=(0, 1.02))
axes[1].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")  # reuse in the report
plt.show()

best = max(val_f1, key=lambda t: t[1])
print(f"Best validation macro-F1 = {best[1]:.3f} at epoch {best[0]:.0f} (checkpoint restored)")

## 8 · Test-Set Evaluation  ⟵ *separate cell, per submission guidelines*

The 33 test utterances below were **never seen during training or model selection**.
This cell is the single source of truth for the numbers reported in the project report.

In [ ]:
# ============================================================
#  FINAL EVALUATION ON THE HELD-OUT TEST SET (run exactly once)
# ============================================================
from sklearn.metrics import accuracy_score, classification_report

test_out = trainer.predict(test_ds)
y_pred = test_out.predictions.argmax(-1)

acc = accuracy_score(y_test, y_pred)
p, r, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro", zero_division=0)

print(f"TEST accuracy      : {acc:.3f}")
print(f"TEST macro precision: {p:.3f}")
print(f"TEST macro recall  : {r:.3f}")
print(f"TEST macro F1      : {f1:.3f}")
print(f"TEST loss          : {test_out.metrics['test_loss']:.4f}\n")

print(classification_report(y_test, y_pred,
                            target_names=label_list, zero_division=0))

## 9 · Error Analysis

The confusion matrix shows *which* intents get mixed up — in the domain, `refine`
("show me more like that") and `recommend` share vocabulary, and `oos` is the open class,
so those borders are where errors are expected.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=label_list,
    cmap="Blues", colorbar=False, xticks_rotation=45, ax=ax)
ax.set_title("Confusion matrix — test set")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# Inspect every misclassified utterance — with 33 test examples we can read them all
print("Misclassified test utterances:")
n_err = 0
for text, t, pr in zip(X_test, y_test, y_pred):
    if t != pr:
        n_err += 1
        print(f"  '{text}'  true={id2label[int(t)]:<10} pred={id2label[int(pr)]}")
if n_err == 0:
    print("  (none — perfect test-set classification)")

## 10 · End-to-End Inference Demo

The full CineBot pipeline pairs this classifier with rule-based **entity extraction**
(mood / genre / era — see `app/nlp/entity_extractor.py` in the repo) before querying the hybrid
recommender. The compact version below shows how a raw user message becomes structured
*chatbot understanding*.

In [ ]:
import re

# Compact gazetteer-based entity extraction (full version lives in the repo)
MOODS  = {"sad", "happy", "stressed", "bored", "romantic", "nostalgic", "scared", "excited"}
GENRES = {"comedy", "horror", "thriller", "drama", "action", "romance", "sci-fi",
          "science fiction", "animation", "documentary", "fantasy", "musical"}
ERA_RE = re.compile(r"\b(19[2-9]0|20[0-2]0)s\b")

def extract_entities(text):
    t = text.lower()
    return {
        "mood":  [m for m in MOODS if re.search(rf"\b{m}\b", t)],
        "genre": [g for g in GENRES if g in t],
        "era":   ERA_RE.findall(t),
    }

def understand(message):
    """Classify intent (fine-tuned DistilBERT) + extract entities (rules)."""
    enc = tokenizer(message, truncation=True, padding="max_length",
                    max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = trainer.model(**enc).logits
    probs = torch.softmax(logits, dim=-1)[0]
    conf, idx = probs.max(dim=-1)
    return {"intent": id2label[int(idx)], "confidence": round(float(conf), 3),
            "entities": extract_entities(message)}

for msg in ["hey there!",
            "I'm feeling sad, suggest something feel-good from the 1990s",
            "show me more like that one",
            "what's the plot of that movie?",
            "I loved it, thanks!",
            "what's the capital of France?"]:
    print(f"  '{msg}'\n    -> {understand(msg)}\n")

In [ ]:
# Persist the fine-tuned model + tokenizer (same layout the FastAPI service loads)
OUT = Path("ml/models/intent_classifier")
OUT.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUT))
tokenizer.save_pretrained(str(OUT))
print("Saved serving-ready model to", OUT)

## 11 · Conclusions & Future Work

**What worked**
- Transfer learning is decisive here: fine-tuning a 66M-parameter pretrained encoder on ~150 training
  utterances reaches high macro-F1 — training the same architecture from scratch on this data would be hopeless.
- Stratified splitting + macro-F1 model selection handled the class imbalance without resampling.
- Best-checkpoint restoration on validation F1 acted as effective early stopping against overfitting.

**What didn't / limitations**
- The dataset (216 examples) is the bottleneck — confusions concentrate where vocabulary overlaps
  (`refine` vs `recommend`) and in the open-ended `oos` class.
- Entity extraction is rule-based; it is precise on the curated vocabulary but cannot generalise to
  unseen phrasings ("something tear-jerky").

**Future improvements**
1. Grow the dataset to ~500 utterances (the repo's stated target), prioritising `oos` and `refine`.
2. Replace gazetteer NER with a learned token-classification head once ≥100 annotated utterances exist.
3. Add an NDCG@5 offline evaluation of the recommender against a hand-labelled query set.
4. Calibrate the confidence threshold for the `oos` fallback using the validation set.